# Skincare Dupe Finder

## Data Scraping

Objective:

Collect skincare product information from INCIDecoder including:
- Product name
- Ingredient list
- Product URL

The scraped data will be stored as a raw dataset for later preprocessing and similarity analysis.

In [2]:
!pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

SEED = 42

In [3]:
#### load urls

In [9]:
old_df = pd.read_csv("../../data/skincare_products.csv")

urls = (old_df["url"]
    .dropna()
    .unique()
    .tolist())

print(len(urls))

150


In [11]:
print(old_df.columns)
print(old_df.shape)

Index(['product_name', 'ingredients', 'url'], dtype='str')
(150, 3)


#### Scraper Function

#### 1) Scrape products

In [13]:
def scrape_product(url):
    
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to retrieve {url}")
        return None
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Product Name
    product_name = soup.title.text.replace(" ingredients (Explained)", "").strip()
    
    # Ingredient Section
    short_section = soup.find("div", class_="ingredlist-short-like-section")
    
    if short_section is None:
        print(f"No ingredient section found for {url}")
        return None
    
    ingredients = short_section.find_all("a")
    
    ingredient_names = [
        i.text.strip().lower()
        for i in ingredients
        if "how to read" not in i.text.lower()
    ]
    
    ingredient_string = ", ".join(ingredient_names)
    
    return {
        "product_name": product_name,
        "ingredients": ingredient_string,
        "url": url
    }

In [14]:
scrape_product(urls[0])

{'product_name': 'COSRX Advanced Snail Radiance Dual Essence',
 'ingredients': 'snail secretion filtrate, butylene glycol, niacinamide, 1,2-hexanediol, betaine, panthenol, water, glycerin, limnanthes alba (meadowfoam) seed oil, helianthus annuus (sunflower) seed oil, allantoin, carbomer, macadamia ternifolia seed oil, sodium polyacrylate, sodium hyaluronate, inulin lauryl carbamate, arginine, argania spinosa kernel oil, xanthan gum, sodium surfactin, ethylhexylglycerin, glutathione',
 'url': 'https://incidecoder.com/products/cosrx-advanced-snail-radiance-dual-essence'}

#### scrape all products

In [15]:
products = []
for url in tqdm(urls):
    result = scrape_product(url)
    if result:
        products.append(result)

100%|█████████████████████████████████████████| 150/150 [02:10<00:00,  1.15it/s]


#### create a dataframe

In [16]:
df = pd.DataFrame(products)

In [17]:
df.shape
df.isnull().sum()

product_name    0
ingredients     0
url             0
dtype: int64

In [18]:
df.head()

,product_name,ingredients,url
0,COSRX Advanced Snail Radiance Dual Essence,"snail secretion filtrate, butylene glycol, nia...",https://incidecoder.com/products/cosrx-advance...
1,CeraVe Baby Moisturizing Lotion,"water, cetearyl alcohol, caprylic/​capric trig...",https://incidecoder.com/products/cerave-baby-m...
2,COSRX Ac Collection Calming Foam Cleanser,"water, glycerin, stearic acid, myristic acid, ...",https://incidecoder.com/products/cosrx-ac-coll...
3,The Ordinary Aha 30% + Bha 2% Peeling Solution...,"glycolic acid, aqua (water), aloe barbadensis ...",https://incidecoder.com/products/the-ordinary-...
4,The Ordinary Buffet + Copper Peptides 1%,"aqua (water), glycerin, lactococcus ferment ly...",https://incidecoder.com/products/the-ordinary-...


In [19]:
df.to_csv("../../data/raw_products_new.csv",index=False)

In [20]:
df.shape

(150, 3)

In [21]:
df["product_name"].duplicated().sum()

np.int64(20)